In [ ]:
# Squat motion-disentangling SSL (R(2+1)D-18, triplet distance-ratio pretext).
#
# Run in Colab on an L4 GPU (24 GB VRAM). Mounts the Drive shortcut
# `My Drive/Fitness-AQA_dataset_release` (reused from the earlier squat runs).
#
# Step 1 (the probe) is a human-verify checkpoint: it confirms the unlabeled
# trajectory file format + the argmax-vs-argmin half-cycle SIGN on real clips
# before `squat_ssl._load_trajectory` is finalized and before any GPU
# pretraining. A wrong sign swaps descent/ascent and wastes the 12-24h run.
# Review the probe report + traj_probe.png + the resolved sign after Step 1.
#
# Jupytext "percent" format: each `# %%` marker starts a new cell. Convert via
# `pip install jupytext && jupytext --to ipynb 04_squat_md_ssl.py`, or paste cells
# one at a time.

# Squat Motion-Disentangling Self-Supervised Pretraining

Motion-disentangling SSL for squat form on the Fitness-AQA dataset: an R(2+1)D-18 backbone pretrained with a triplet distance-ratio pretext on 4,970 unlabeled clips and their barbell trajectories, then fine-tuned on the labeled KIE/KFE splits. Run on an L4 GPU (24 GB VRAM).

In [1]:
import os
import subprocess
import sys

REPO_DIR = "/content/fitnova"
REPO_URL = "https://github.com/Ibrahim-Shahin1/fitnova.git"
BRANCH = "fresh-start"

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL} (branch {BRANCH}) -> {REPO_DIR} ...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repo already at {REPO_DIR}; pulling latest from origin/{BRANCH} ...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

_head = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f"\nRepo ready: {REPO_DIR} @ {_head}")
print(f"sys.path[0]: {sys.path[0]}")
print(f"cwd: {os.getcwd()}")

Cloning https://github.com/Ibrahim-Shahin1/fitnova.git (branch fresh-start) -> /content/fitnova ...

Repo ready: /content/fitnova @ f1d9613
sys.path[0]: /content/fitnova
cwd: /content/fitnova


## Setup

Clone the repo onto the Python path, then stage the labeled (1,739) and unlabeled (4,970) squat clips plus barbell trajectories from Drive.

The first import must be `from backend.training.aqa.harness import _envinit`, which sets `CUBLAS_WORKSPACE_CONFIG` before `torch` creates the CUDA context. The dependency probe includes `scipy` (`split_half_cycles` uses `scipy.ndimage`; the trajectory probe uses `scipy.signal.find_peaks`).

In [2]:
from backend.training.aqa.harness import _envinit  # sets CUBLAS_WORKSPACE_CONFIG before torch

import importlib.metadata
import os
import shutil
import subprocess
import sys

# PyAV must be imported before the first torch/torchvision import.
try:
    import av  # noqa: F401
    _pyav_status = "already installed"
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)
    import av  # noqa: F401
    _pyav_status = "installed now"
print(f"PyAV {av.__version__} ({_pyav_status})")

# Dependency version probe (supply-chain guard). Zero new pip installs.
_PHASE4_DEPS = ["torch", "torchvision", "scikit-learn", "matplotlib", "tqdm", "numpy", "scipy"]
print("\nDependency versions (supply-chain guard):")
for _pkg in _PHASE4_DEPS:
    try:
        print(f"  {_pkg:<14} {importlib.metadata.version(_pkg)}")
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError(
            f"Dependency '{_pkg}' is NOT installed. This run introduces "
            "zero new pip dependencies beyond PyAV — every package was already in "
            "backend/requirements.txt. If this fires, the Colab runtime is misconfigured."
        )

import torch
import torchvision

print("\npython     :", sys.version.split()[0])
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUBLAS_WORKSPACE_CONFIG:", os.environ.get("CUBLAS_WORKSPACE_CONFIG"))

# GPU + VRAM check.
assert torch.cuda.is_available(), (
    "GPU required — Runtime > Change runtime type > Hardware accelerator: L4"
)
_gpu_name = torch.cuda.get_device_name(0)
_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"CUDA device : {_gpu_name} ({_vram_gb:.1f} GB VRAM)")
assert "L4" in _gpu_name or _vram_gb >= 22.0, (
    f"Expecting an L4 (>=22 GB); got {_gpu_name} {_vram_gb:.1f} GB. "
    "VRAM sizing assumes 24 GB headroom."
)

# Verify torchvision.io PyAV backend (no real video needed).
from torchvision.io.video import av as _tv_av_ref
assert not isinstance(_tv_av_ref, Exception), (
    "torchvision.io cached `av` as an Exception — PyAV install ordered wrong."
)
print("torchvision.io PyAV backend: ready")

# Drive mount (idempotent).
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'
ROOT = os.path.join(MYDRIVE, 'Fitness-AQA_dataset_release')
print("\nROOT :", ROOT, "| exists:", os.path.exists(ROOT))

# Stage LABELED videos (1739 — reused for the linear-probe monitor).
from pathlib import Path
from backend.training.aqa.harness.colab import stage_squat_videos, stage_unlabeled_squat_videos

VIDEOS_ROOT = stage_squat_videos(MYDRIVE)
_labeled_n = len(list(Path(VIDEOS_ROOT).glob('*.mp4')))
print(f"\nLABELED VIDEOS_ROOT: {VIDEOS_ROOT} | mp4 count: {_labeled_n} (expected 1739)")
assert _labeled_n == 1739, f"Expected 1739 labeled mp4s, got {_labeled_n}"

# Log the unlabeled videos.zip size + free /content disk BEFORE staging the
# 4,970-clip set (it is ~4x the labeled set — confirm there is room).
_unlabeled_zip = Path(ROOT) / "Squat/Unlabeled_Dataset/videos.zip"
if _unlabeled_zip.is_file():
    print(f"\nunlabeled videos.zip: {_unlabeled_zip.stat().st_size / 1e9:.2f} GB")
_free_gb = shutil.disk_usage("/content").free / 1e9
print(f"free /content disk: {_free_gb:.1f} GB")

# Stage UNLABELED videos + barbell trajectories (4,970 — the SSL pretrain set).
UNLABELED_VIDEOS_ROOT, TRAJ_ROOT = stage_unlabeled_squat_videos(MYDRIVE)
_unlabeled_n = len(list(Path(UNLABELED_VIDEOS_ROOT).glob('*.mp4')))
print(f"\nUNLABELED_VIDEOS_ROOT: {UNLABELED_VIDEOS_ROOT} | mp4 count: {_unlabeled_n} (expected 4970)")
print(f"TRAJ_ROOT: {TRAJ_ROOT}")
assert _unlabeled_n == 4970, f"Expected 4970 unlabeled mp4s, got {_unlabeled_n}"

PyAV 17.0.1 (installed by Step 0)

[T-04-04] Dependency versions (supply-chain guard):
  torch          2.10.0+cu128
  torchvision    0.25.0+cu128
  scikit-learn   1.6.1
  matplotlib     3.10.0
  tqdm           4.67.3
  numpy          2.0.2
  scipy          1.16.3

python     : 3.12.13
torch      : 2.10.0+cu128
torchvision: 0.25.0+cu128
CUDA available: True
CUBLAS_WORKSPACE_CONFIG: :4096:8
CUDA device : NVIDIA L4 (22.0 GB VRAM)
torchvision.io PyAV backend: ready
Mounted at /content/drive

ROOT : /content/drive/MyDrive/Fitness-AQA_dataset_release | exists: True


copy videos.zip:   0%|          | 0.00/925M [00:00<?, ?B/s]

unzip squat_videos.zip:   0%|          | 0/1739 [00:00<?, ?file/s]


LABELED VIDEOS_ROOT: /content/squat_videos | mp4 count: 1739 (expected 1739)

[T-04-08] unlabeled videos.zip: 1.68 GB
[T-04-08] free /content disk: 199.9 GB


copy videos.zip:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

unzip squat_unlabeled_videos.zip:   0%|          | 0/4970 [00:00<?, ?file/s]

copy bar_trajectories_raw.zip:   0%|          | 0.00/1.77M [00:00<?, ?B/s]


UNLABELED_VIDEOS_ROOT: /content/squat_unlabeled_videos | mp4 count: 4970 (expected 4970)
TRAJ_ROOT: /content/squat_trajectories


## Step 1 — Trajectory format and half-cycle-sign probe

Confirms the unlabeled barbell-trajectory file format and the argmax-vs-argmin sign that `split_half_cycles` uses to place the rep-bottom (deepest squat), before the SSL run. A wrong sign swaps descent and ascent.

The probe reports archive structure, value type, raw-vs-smoothed signal, trajectory-length-vs-frame-count mapping, and ID alignment; overlays both sign values on three sample clips with decoded descent/ascent frame grids; and counts multi-rep frequency across all 4,970 trajectories with `scipy.signal.find_peaks`.

In [ ]:
import glob
import json

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage
import torch
import torchvision
from scipy.signal import find_peaks

from backend.training.aqa.datasets.squat_ssl import split_half_cycles
from backend.training.aqa.datasets.transforms import decode_clip

FIG_DIR = Path("docs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _extract_y_series(data):
    """Defensively pull a 1-D y-trajectory out of whatever the JSON holds.

    Returns (y_series: np.ndarray, how: str) so the probe REPORTS the path taken
    rather than silently assuming a format.
    """
    arr = np.asarray(data, dtype=object)
    if isinstance(data, list) and data and isinstance(data[0], (int, float)):
        return np.asarray(data, dtype=float), "flat list of floats (== [ASSUMED])"
    if isinstance(data, list) and data and isinstance(data[0], (list, tuple)):
        cols = len(data[0])
        col = 1 if cols in (2, 4) else 0  # [x,y] or [x,y,w,h] -> y at index 1
        return np.asarray([row[col] for row in data], dtype=float), f"list of {cols}-tuples -> col {col}"
    if isinstance(data, dict):
        for key in ("y", "y_center", "cy", "centers", "trajectory", "traj"):
            if key in data:
                return _extract_y_series(data[key])[0], f"dict['{key}']"
        return np.asarray([], dtype=float), f"dict with keys {list(data.keys())[:8]} (UNHANDLED — describe below)"
    return np.asarray(arr, dtype=float).ravel(), "fallback ravel (UNHANDLED — describe below)"


# ── (1) Archive internal structure ───────────────────────────
all_traj = sorted(p for p in glob.glob(os.path.join(TRAJ_ROOT, "**", "*"), recursive=True) if os.path.isfile(p))
exts = {}
for p in all_traj:
    exts[Path(p).suffix.lower()] = exts.get(Path(p).suffix.lower(), 0) + 1
print(f"[structure] {len(all_traj)} trajectory files under {TRAJ_ROOT}")
print(f"  extensions: {exts}")
print(f"  sample names: {[os.path.basename(p) for p in all_traj[:5]]}")

# ── (2)+(3) Value type + raw/smoothed (load up to 5) ──────────────────────
json_files = [p for p in all_traj if p.lower().endswith(".json")]
sample_files = json_files[:5] if json_files else all_traj[:5]
print(f"\n[value type] inspecting {len(sample_files)} files:")
parsed = []  # (clip_id, y_series, how)
for p in sample_files:
    try:
        with open(p) as fh:
            data = json.load(fh)
        y, how = _extract_y_series(data)
        parsed.append((Path(p).stem, y, how))
        print(f"  {Path(p).name}: top-type={type(data).__name__} -> y via {how}; "
              f"len={len(y)}, y[:3]={np.round(y[:3], 2).tolist()}, "
              f"range=[{y.min():.1f},{y.max():.1f}]" if len(y) else f"  {Path(p).name}: EMPTY/UNHANDLED ({how})")
    except Exception as e:  # noqa: BLE001
        print(f"  {Path(p).name}: NOT JSON or load error -> {e!r} (describe the real format below)")

# ── (5) ID alignment: do traj stems match clip IDs 1:1? ───────────────────
traj_stems = {Path(p).stem for p in json_files} if json_files else {Path(p).stem for p in all_traj}
vid_stems = {p.stem for p in Path(UNLABELED_VIDEOS_ROOT).glob("*.mp4")}
print(f"\n[ID alignment] traj stems={len(traj_stems)} vid stems={len(vid_stems)} "
      f"| intersect={len(traj_stems & vid_stems)} | traj-only={len(traj_stems - vid_stems)} "
      f"| vid-only={len(vid_stems - traj_stems)}")

# ── (6) traj_nan list present in the unlabeled set? ───────────────────────
_nan_candidates = [p for p in all_traj if "nan" in os.path.basename(p).lower()]
print(f"[traj_nan] files with 'nan' in name: {[os.path.basename(p) for p in _nan_candidates[:5]]} "
      f"(total {len(_nan_candidates)})")

# ── (4) traj length vs video frame count (mapping) for the sample clips ───────
import torchvision
print(f"\n[traj->frame mapping] (traj_len vs video_frame_count per clip):")
for clip_id, y, _how in parsed:
    if len(y) == 0:
        continue
    vpath = os.path.join(UNLABELED_VIDEOS_ROOT, f"{clip_id}.mp4")
    if not os.path.isfile(vpath):
        print(f"  {clip_id}: video not found at {vpath}")
        continue
    try:
        pts, _fps = torchvision.io.read_video_timestamps(vpath, pts_unit="sec")
        n_frames = len(pts)
        ratio = n_frames / len(y) if len(y) else float("nan")
        print(f"  {clip_id}: traj_len={len(y)}  video_frames={n_frames}  frames/traj={ratio:.3f} "
              f"({'1:1' if abs(ratio - 1.0) < 0.05 else 'NOT 1:1 — must rescale traj->frame'})")
    except Exception as e:  # noqa: BLE001
        print(f"  {clip_id}: read_video_timestamps failed -> {e!r}")

# ── SIGN DECISION figure: split overlay (both signs) + descent/ascent frame grids ──
n_show = min(3, sum(1 for _, y, _ in parsed if len(y) >= 4))
fig = plt.figure(figsize=(16, 4 * max(n_show, 1)))
row = 0
for clip_id, y, _how in parsed:
    if len(y) < 4:
        continue
    if row >= n_show:
        break
    sm = None
    try:
        import scipy.ndimage
        sm = scipy.ndimage.gaussian_filter1d(y, sigma=2.0)
    except Exception:
        sm = y
    b_max, b_min = int(np.argmax(sm)), int(np.argmin(sm))

    # Trajectory plot with BOTH candidate bottoms.
    ax = fig.add_subplot(n_show, 3, row * 3 + 1)
    ax.plot(y, color="0.6", lw=1, label="raw y")
    ax.plot(sm, color="C0", lw=2, label="smoothed")
    ax.axvline(b_max, color="C3", ls="--", label=f"argmax bottom @{b_max}")
    ax.axvline(b_min, color="C2", ls=":", label=f"argmin bottom @{b_min}")
    ax.set_title(f"{clip_id}  (len {len(y)})")
    ax.legend(fontsize=7)

    # Descent/ascent frame strips for the argmax sign (the [ASSUMED] default).
    vpath = os.path.join(UNLABELED_VIDEOS_ROOT, f"{clip_id}.mp4")
    try:
        desc, asc = split_half_cycles(y, frames_per_half=8, bottom_is_argmax=True)
        for j, (idxs, label, sub) in enumerate([(desc, "DESCENT (argmax)", 2), (asc, "ASCENT (argmax)", 3)]):
            clip = decode_clip(vpath, torch.as_tensor(np.asarray(idxs), dtype=torch.long))  # [T,3,H,W] uint8
            strip = torch.cat([clip[k] for k in range(min(8, clip.shape[0]))], dim=2)  # concat along W
            axg = fig.add_subplot(n_show, 3, row * 3 + sub)
            axg.imshow(strip.permute(1, 2, 0).numpy())
            axg.set_title(f"{label}  idx {np.asarray(idxs)[:4].tolist()}...", fontsize=8)
            axg.axis("off")
    except Exception as e:  # noqa: BLE001
        print(f"  [frame grid] {clip_id} failed: {e!r}")
    row += 1

fig.suptitle("Trajectory probe — confirm which sign puts the split at the rep-bottom (deepest squat)\n"
             "argmax (red --) vs argmin (green :) — and which makes DESCENT frames go DOWN", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.96))
_probe_png = FIG_DIR / "traj_probe.png"
fig.savefig(_probe_png, dpi=110, bbox_inches="tight")  # save BEFORE show (disconnect-safe)
print(f"\n[figure] saved {_probe_png}")
plt.show()

# ── multi-rep frequency across ALL 4,970 trajectories ──
import scipy.ndimage
multi_rep = 0
total_ok = 0
for p in (json_files or all_traj):
    try:
        with open(p) as fh:
            y, _ = _extract_y_series(json.load(fh))
        if len(y) < 4:
            continue
        sm = scipy.ndimage.gaussian_filter1d(np.asarray(y, dtype=float), sigma=2.0)
        rng = sm.max() - sm.min()
        if rng <= 0:
            continue
        peaks, _ = find_peaks(sm, prominence=0.3 * rng)
        valleys, _ = find_peaks(-sm, prominence=0.3 * rng)
        total_ok += 1
        if (len(peaks) + len(valleys)) > 1:
            multi_rep += 1
    except Exception:  # noqa: BLE001
        continue
print(f"\n[multi-rep] {multi_rep}/{total_ok} trajectories have >1 prominent extremum "
      f"({100 * multi_rep / max(total_ok, 1):.1f}% — single-rep argmax/argmin covers the rest; "
      "find_peaks splitting is added only if this fraction is large)")

print("\n=== Probe complete: review this report + traj_probe.png + the resolved sign. ===")

## Step 2 — SSL dataset smoke test

Instantiates `SquatSSLDataset`, pulls one batch via `build_ssl_loader`, and asserts the three `(B, 3, 16, 112, 112)` float32 triplet tensors (anchor, positive, negative). Renders an 8-frame strip of the raw descent (anchor) and ascent (negative) half-cycles as a second visual check of the split through the full data pipeline.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import torch

import backend.training.aqa.datasets.squat_ssl as _squat_ssl
importlib.reload(_squat_ssl)  # refresh after git pull (sys.modules is stale)
from backend.training.aqa.datasets.squat_ssl import SquatSSLDataset, build_ssl_loader
from backend.training.aqa.harness.md_pretrain import MDConfig

_FIG_DIR = Path("docs/figures")
_FIG_DIR.mkdir(parents=True, exist_ok=True)

config = MDConfig()
ds = SquatSSLDataset(
    videos_root=UNLABELED_VIDEOS_ROOT,
    trajectories_root=TRAJ_ROOT,
    frames_per_half=config.frames_per_half,
    crop_size=config.crop_size,
)
print(f"len(ds) = {len(ds)} (expect ~4970)")

loader = build_ssl_loader(ds, config, seed=42)
batch = next(iter(loader))
_expected = (config.batch_size, 3, config.frames_per_half, config.crop_size, config.crop_size)
for _k in ("anchor", "positive", "negative"):
    _t = batch[_k]
    print(f"  {_k:>8}: shape={tuple(_t.shape)} dtype={_t.dtype}")
    assert tuple(_t.shape) == _expected, (_k, tuple(_t.shape), _expected)
    assert _t.dtype == torch.float32, (_k, _t.dtype)
print("triplet shapes ok")

# Sanity grid: decode the RAW descent/ascent for one clip (NO augs / NO temporal-reverse) so the
# half-cycle split is cleanly visible. The batch's anchor/negative are augmented AND randomly
# temporal-reversed (the global-motion equalizer), which would obscure the down/up direction —
# so the shapes are asserted from the batch (above), but the SPLIT is shown from the raw frames.
from backend.training.aqa.datasets.squat_ssl import split_half_cycles
from backend.training.aqa.datasets.transforms import decode_clip

_cid = ds._clip_ids[0]
_traj = ds._load_trajectory(_cid)
_desc, _asc = split_half_cycles(_traj, frames_per_half=config.frames_per_half, bottom_is_argmax=False)
_vp = os.path.join(UNLABELED_VIDEOS_ROOT, f"{_cid}.mp4")
_desc_u8 = decode_clip(_vp, torch.as_tensor(_desc, dtype=torch.long))  # [16,3,H,W] uint8
_asc_u8 = decode_clip(_vp, torch.as_tensor(_asc, dtype=torch.long))


def _u8_strip(clip_tchw: torch.Tensor, n: int = 8) -> np.ndarray:
    """[T,3,H,W] uint8 -> [H, n*W, 3] uint8 (n evenly-spaced frames) for imshow."""
    sel = torch.linspace(0, clip_tchw.shape[0] - 1, n).round().long()
    return np.concatenate([clip_tchw[t].permute(1, 2, 0).numpy() for t in sel], axis=1)


fig, axes = plt.subplots(2, 1, figsize=(16, 5))
axes[0].imshow(_u8_strip(_desc_u8))
axes[0].set_title(f"{_cid} DESCENT (frames {_desc[:4].tolist()}...) — lifter should be going DOWN")
axes[0].axis("off")
axes[1].imshow(_u8_strip(_asc_u8))
axes[1].set_title(f"{_cid} ASCENT (frames {_asc[:4].tolist()}...) — lifter should be going UP")
axes[1].axis("off")
fig.suptitle("SSL triplet sanity — RAW descent/ascent (re-confirms argmin on real data; batch shapes asserted above)")
fig.tight_layout()
_sanity_png = _FIG_DIR / "ssl_triplet_sanity.png"
fig.savefig(_sanity_png, dpi=110, bbox_inches="tight")  # save BEFORE show (disconnect-safe)
print(f"saved {_sanity_png}")
plt.show()

print("\n=== Smoke test complete: len(ds), the 3 shapes, + ssl_triplet_sanity.png. ===")

## Step 3 — VRAM probe

One forward+backward pass of the 3-branch triplet at batch size 8, then `max_memory_allocated`, to confirm peak memory fits the L4 budget before the full run.

In [ ]:
import importlib
import torch

import backend.training.aqa.datasets.squat_ssl as _sslds
import backend.training.aqa.harness.md_pretrain as _mdp
importlib.reload(_sslds)  # pick up the recursive-glob fix after git pull
importlib.reload(_mdp)     # pick up the finalized trainer after git pull
from backend.training.aqa.datasets.squat_ssl import SquatSSLDataset, build_ssl_loader
from backend.training.aqa.harness.md_pretrain import MDConfig, build_md_model, md_triplet_loss

config = MDConfig()
_dev = torch.device("cuda")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

_bb, _proj = build_md_model()
_bb, _proj = _bb.to(_dev), _proj.to(_dev)
_ds = SquatSSLDataset(videos_root=UNLABELED_VIDEOS_ROOT, trajectories_root=TRAJ_ROOT,
                      frames_per_half=config.frames_per_half, crop_size=config.crop_size)
_b = next(iter(build_ssl_loader(_ds, config, seed=42)))
_opt = torch.optim.AdamW(list(_bb.parameters()) + list(_proj.parameters()),
                         lr=config.learning_rate, weight_decay=config.weight_decay)
_bb.train(); _proj.train()
_pa = _proj(_bb(_b["anchor"].to(_dev)))
_pp = _proj(_bb(_b["positive"].to(_dev)))
_pn = _proj(_bb(_b["negative"].to(_dev)))
_loss = md_triplet_loss(_pa, _pp, _pn, squared=config.loss_squared, three_term=config.loss_three_term)
_opt.zero_grad(); _loss.backward(); _opt.step()
_peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"batch_size={config.batch_size} | peak backward VRAM = {_peak_gb:.2f} GB | loss = {_loss.item():.4f}")
assert _peak_gb < 22.0, f"VRAM {_peak_gb:.1f} GB exceeds the L4 budget — reduce the batch size"
print(f"VRAM fits L4 (< 22 GB) at batch {config.batch_size}")
del _bb, _proj, _opt, _pa, _pp, _pn, _loss, _b
torch.cuda.empty_cache()

## Step 4 — epoch-0 timing probe

One epoch under the production trainer to estimate total SSL training time. `epoch_wall_time_s` is the pure SSL-pass time (measured before the linear probe), so `estimated_total_h = ssl_epoch_time * max_epochs`; the linear probe adds a smaller increment every 5 epochs.

In [ ]:
import os
import time

import torch

from backend.training.aqa.harness.md_pretrain import MDConfig, run_md_pretrain_epoch

config = MDConfig()
_t_cell = time.perf_counter()
_res = run_md_pretrain_epoch(
    run_name="md_pretrain_v1_timing",
    drive_root=MYDRIVE,
    videos_root=UNLABELED_VIDEOS_ROOT,
    trajectories_root=TRAJ_ROOT,
    labeled_videos_root=VIDEOS_ROOT,
    seed=42, config=config, resume=False, max_epochs=1,
)
_cell_wall = time.perf_counter() - _t_cell
_e0 = _res["metrics_history"][0]
_ssl_t = _e0["epoch_wall_time_s"]
_est_h = _ssl_t * config.max_epochs / 3600.0
print(f"epoch-0 SSL pass = {_ssl_t:.1f}s | full cell (incl. linear-probe) = {_cell_wall:.1f}s")
print(f"ssl_loss = {_e0['ssl_loss_mean']:.4f} | embedding_std = {_e0['embedding_std']:.5f} "
      f"(healthy ~{1/512**0.5:.5f}; collapse -> 0) | effective_rank = {_e0['effective_rank']:.1f}")
print(f"estimated SSL training time (x{config.max_epochs} epochs) = {_est_h:.1f} h "
      f"(+ linear-probe every {config.linear_probe_cadence} epochs)")
if _res["linear_probe_history"]:
    _lp = _res["linear_probe_history"][0]
    print(f"epoch-0 linear-probe macro-F1 = {_lp['linear_probe_f1_macro']:.4f} "
          f"(kie={_lp['linear_probe_f1_kie']:.4f} kfe={_lp['linear_probe_f1_kfe']:.4f})")

# Checkpoint round-trip verify.
_ckpt = _res["checkpoint_path"]
print(f"\ncheckpoint: {_ckpt} ({os.path.getsize(_ckpt) / 1e6:.1f} MB)")
_pl = torch.load(_ckpt, map_location="cpu", weights_only=False)
assert _pl["code_version"] == "phase04-md-pretrain", _pl.get("code_version")
print("checkpoint round-trip OK; payload keys:", sorted(_pl.keys()))

print(f"\n=== Timing estimate: est_total≈{_est_h:.1f}h, VRAM fits. "
      "batch 8 if the total time is acceptable; batch 5 if VRAM is tight; "
      "TorchCodec only if GPU util is low (decode-bound). ===")

## Step 5 — full MD-SSL pretraining

Batch 8, 60-epoch cosine schedule, with a linear probe every 5 epochs. `resume=True` checkpoints every epoch to Drive and writes `backbone.pt` whenever the linear-probe macro-F1 improves. SSL health is tracked by a decreasing loss, `embedding_std` staying above 0.1/sqrt(512) (collapse guard), and effective rank staying above 1; the linear-probe macro-F1 rising then plateauing marks convergence.

In [ ]:
import importlib

import backend.training.aqa.harness.md_pretrain as _mdp
importlib.reload(_mdp)  # pick up the finalized trainer + the collapse-metric fix after git pull
from backend.training.aqa.harness.md_pretrain import MDConfig, run_md_pretrain_epoch

config = MDConfig()  # batch 8, 60-epoch cosine, linear_probe_cadence=5
RUN_NAME = "md_pretrain_v1"
result = run_md_pretrain_epoch(
    run_name=RUN_NAME,
    drive_root=MYDRIVE,
    videos_root=UNLABELED_VIDEOS_ROOT,
    trajectories_root=TRAJ_ROOT,
    labeled_videos_root=VIDEOS_ROOT,
    seed=42, config=config, resume=True, max_epochs=config.max_epochs,
)

print(f"\nfinal epoch = {result['epoch']} | collapsed = {result['collapsed']}")
print(f"backbone.pt = {result['backbone_path']}")
print("\nlinear-probe history (watch macro rise then plateau — the convergence stop):")
for _e in result["linear_probe_history"]:
    print(f"  epoch {_e['epoch']:>2}: macro={_e['linear_probe_f1_macro']:.4f} "
          f"(kie={_e['linear_probe_f1_kie']:.4f} kfe={_e['linear_probe_f1_kfe']:.4f})")
print("\nssl_loss + collapse trend (loss down; emb_std not -> 0; eff_rank not -> 1):")
for _m in result["metrics_history"]:
    print(f"  epoch {_m['epoch']:>2}: ssl_loss={_m['ssl_loss_mean']:.4f} "
          f"emb_std={_m['embedding_std']:.5f} eff_rank={_m['effective_rank']:.1f}")

In [3]:
# The SSL output isn't gone — it's on Drive. This reprints it from the checkpoint.
from google.colab import drive; drive.mount('/content/drive')
import os, torch
run_dir = "/content/drive/MyDrive/FitNova/checkpoints/phase04/md_pretrain_v2"
print("files on Drive:", sorted(os.listdir(run_dir)))
with open(os.path.join(run_dir, "latest.txt")) as f: name = f.read().strip()
ck = torch.load(os.path.join(run_dir, name), map_location="cpu", weights_only=False)
print(f"\nrecovered {name} (last epoch {ck['epoch']})")
print("v2 linear-probe curve:")
for e in ck["linear_probe_history"]:
    print(f"  ep {e['epoch']:>2}: macro={e['linear_probe_f1_macro']:.4f} "
          f"(kie={e['linear_probe_f1_kie']:.4f} kfe={e['linear_probe_f1_kfe']:.4f})")
bb = os.path.join(run_dir, "backbone.pt")
print(f"\nbackbone.pt: {os.path.isfile(bb)} ({os.path.getsize(bb)/1e6:.0f} MB) — fine-tune loads this")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
files on Drive: ['backbone.pt', 'epoch_017.pt', 'epoch_018.pt', 'epoch_019.pt', 'latest.txt']

recovered epoch_019.pt (last epoch 19)
v2 linear-probe curve:
  ep  0: macro=0.5143 (kie=0.3299 kfe=0.6987)
  ep  5: macro=0.5844 (kie=0.4554 kfe=0.7133)
  ep 10: macro=0.5443 (kie=0.3486 kfe=0.7400)
  ep 15: macro=0.5605 (kie=0.4000 kfe=0.7211)

backbone.pt: True (380 MB) — fine-tune loads this


In [4]:
# ═══ Fine-tune — setup + sanity ═══
# ⚠️ Re-run the setup cell first (pulls the new md_finetune code), then re-run the staging cell on a fresh runtime.
import importlib, os, torch
import backend.training.aqa.harness.supervised_train as _sup; importlib.reload(_sup)
import backend.training.aqa.harness.md_finetune as _mft;     importlib.reload(_mft)
from torch.utils.data import DataLoader
from backend.training.aqa.datasets.squat import SquatKIEKFEDataset

MD_BACKBONE = os.path.join(MYDRIVE, "FitNova/checkpoints/phase04/md_pretrain_v2/backbone.pt")
assert os.path.isfile(MD_BACKBONE), f"backbone not found: {MD_BACKBONE}"
_ck = torch.load(MD_BACKBONE, map_location="cpu", weights_only=False)
print(f"MD backbone = md_pretrain_v2/backbone.pt (epoch={_ck['epoch']}, expect 5)")

device = torch.device("cuda")
model = _mft.build_finetune_model(MD_BACKBONE, dropout=0.2).to(device).eval()
print("fine-tune head:", model.fc)   # expect Sequential(Dropout(p=0.2), Linear(512->2))

# sanity: one real labeled VAL batch -> logits [B, 2]
ds = SquatKIEKFEDataset(split="val", train_aug=False, drive_root=MYDRIVE,
                        videos_root=VIDEOS_ROOT, num_frames=32, crop_size=112)
clip, label = next(iter(DataLoader(ds, batch_size=4, num_workers=0, shuffle=False)))
with torch.no_grad():
    logits = model(clip.to(device))
print(f"forward OK: clip {tuple(clip.shape)} -> logits {tuple(logits.shape)} (expect (4, 2))")
print(f"val set size={len(ds)}  pos_weight(KIE,KFE)={ds.pos_weight.tolist()}")

MD backbone = md_pretrain_v2/backbone.pt (epoch=5, expect 5)
fine-tune head: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=512, out_features=2, bias=True)
)
forward OK: clip (4, 3, 32, 112, 112) -> logits (4, 2) (expect (4, 2))
val set size=243  pos_weight(KIE,KFE)=[6.099999904632568, 0.4526854157447815]


In [5]:
# ═══ seed 42 timing probe (1 real epoch, resume-safe) ═══
config = _mft.FinetuneConfig(max_epochs=50)   # AdamW lr1e-4 wd1e-4 dropout0.2, 50ep/8-patience, cosine T_max=50
res = _mft.run_md_finetune_epoch(
    run_name="md_finetune_seed42",
    md_backbone_path=MD_BACKBONE,
    drive_root=MYDRIVE, videos_root=VIDEOS_ROOT,
    seed=42, config=config, resume=True, max_epochs=1,   # PROBE: epoch 0 only; the continue cell resumes to 50
)
m0 = res["metrics_history"][-1]
print(f"\nepoch {m0['epoch']}: wall={m0['epoch_wall_time_s']:.0f}s")
print(f"  train_loss={m0['train_loss_mean']:.4f}  val_loss={m0['val_loss_mean']:.4f}  "
      f"train/val ratio={m0['train_val_loss_ratio']:.2f}  (aborts if >10 before ep10)")
print(f"  val_macro_f1={m0['val_macro_f1']:.4f} (kie={m0['val_f1_kie']:.4f} kfe={m0['val_f1_kfe']:.4f})")
print(f"  aborted_overfit={res['aborted_overfit']}")
_w = m0['epoch_wall_time_s']
print(f"\nest per-seed (~13 epochs w/ early-stop) ≈ {_w*13/60:.0f} min; 3 seeds ≈ {_w*13*3/60:.0f} min")

seed42 epoch 0/0 train:   0%|          | 0/71 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]


epoch 0: wall=364s
  train_loss=0.7890  val_loss=0.7727  train/val ratio=1.02  (D6 aborts if >10 before ep10)
  val_macro_f1=0.5462 (kie=0.3624 kfe=0.7300)
  aborted_overfit=False

est per-seed (~13 epochs w/ early-stop) ≈ 79 min; 3 seeds ≈ 237 min


In [6]:
# ═══ seed 42 full fine-tune (resume ep1 -> 50, self early-stops) ═══
config = _mft.FinetuneConfig(max_epochs=50)   # same config as the probe (config_hash must match to resume)
res42 = _mft.run_md_finetune_epoch(
    run_name="md_finetune_seed42",
    md_backbone_path=MD_BACKBONE,
    drive_root=MYDRIVE, videos_root=VIDEOS_ROOT,
    seed=42, config=config, resume=True, max_epochs=50,
)
print(f"\nseed 42 DONE: final epoch={res42['epoch']}  best_val_macro_f1={res42['best_f1_val']:.4f}  "
      f"aborted_overfit={res42['aborted_overfit']}")
print(f"best.pt = {res42['best_checkpoint_path']}")
print("\nval macro-F1 curve (early-stop peak = the best.pt epoch):")
for m in res42["metrics_history"]:
    print(f"  ep {m['epoch']:>2}: train={m['train_loss_mean']:.3f} val={m['val_loss_mean']:.3f} "
          f"ratio={m['train_val_loss_ratio']:.2f} macro={m['val_macro_f1']:.4f} "
          f"(kie={m['val_f1_kie']:.4f} kfe={m['val_f1_kfe']:.4f})")

seed42 epoch 1/49 train:   0%|          | 0/71 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a226478dee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a226478dee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

val:   0%|          | 0/16 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a226478dee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a226478dee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

seed42 epoch 2/49 train:   0%|          | 0/71 [00:00<?, ?it/s]

...

seed42 epoch 15/49 train:   0%|          | 0/71 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]

seed42 epoch 16/49 train:   0%|          | 0/71 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]


seed 42 DONE: final epoch=16  best_val_macro_f1=0.6062  aborted_overfit=False
best.pt = /content/drive/MyDrive/FitNova/checkpoints/phase04/md_finetune_seed42/best.pt

val macro-F1 curve (early-stop peak = the best.pt epoch):
  ep  0: train=0.789 val=0.773 ratio=1.02 macro=0.5462 (kie=0.3624 kfe=0.7300)
  ep  1: train=0.666 val=0.722 ratio=0.92 macro=0.5595 (kie=0.3761 kfe=0.7429)
  ep  2: train=0.689 val=0.770 ratio=0.90 macro=0.5569 (kie=0.3740 kfe=0.7398)
  ep  3: train=0.624 val=0.726 ratio=0.86 macro=0.5682 (kie=0.4231 kfe=0.7133)
  ep  4: train=0.561 val=0.705 ratio=0.80 macro=0.5969 (kie=0.4286 kfe=0.7653)
  ep  5: train=0.460 val=1.075 ratio=0.43 macro=0.5334 (kie=0.3797 kfe=0.6871)
  ep  6: train=0.418 val=1.146 ratio=0.36 macro=0.5595 (kie=0.3571 kfe=0.7619)
  ep  7: train=0.357 val=1.543 ratio=0.23 macro=0.5553 (kie=0.3810 kfe=0.7297)
  ep  8: train=0.331 val=1.127 ratio=0.29 macro=0.6062 (kie=0.4112 kfe=0.8011)
  ep  9: train=0.337 val=1.012 ratio=0.33 macro=0.5228 (kie=0.3